# 02 — Resume Parser + Section Splitter + Chunking

Parse extracted resume text into structured sections, then chunk each section into
smaller, semantically meaningful units with attached metadata.

**Input:** `data/processed/resumes_extracted.json` (from notebook 01)  
**Outputs:**
- `data/processed/resumes_parsed.json` — per-resume structured section breakdown
- `data/processed/resume_chunks.json` — flat list of chunks with metadata, ready for embedding

### What this notebook does
1. **Section detection** — regex + heuristic rules identify headers like *Experience*, *Education*, *Skills*, *Projects*, *Certifications*, *Summary*
2. **Contact extraction** — pull name, email, phone, LinkedIn, GitHub from the header block
3. **Per-section parsing** — experience entries get structured into `{title, company, dates, bullets}`, education into `{degree, institution, dates}`, skills into a flat list
4. **Chunking** — each section is split into overlapping or non-overlapping chunks tagged with `section_type` and rich metadata

In [ ]:
import json
import re
import uuid
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from tqdm import tqdm

PROJECT_ROOT = Path("../").resolve()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

INPUT_PATH   = PROCESSED_DIR / "resumes_extracted.json"
PARSED_PATH  = PROCESSED_DIR / "resumes_parsed.json"
CHUNKS_PATH  = PROCESSED_DIR / "resume_chunks.json"

print(f"Input  : {INPUT_PATH}")
print(f"Outputs: {PARSED_PATH}")
print(f"         {CHUNKS_PATH}")

## 1 — Section Detection

We use regex patterns to identify common resume section headers.  
Sections are detected by matching lines that look like headers (short, uppercase or title-cased, often followed by a separator).

In [ ]:
# Section header patterns — ordered from most to least specific
SECTION_PATTERNS: List[Tuple[str, re.Pattern]] = [
    ("contact",        re.compile(r"^(contact|personal\s+info(rmation)?|about\s+me)\s*[:\-]?", re.I)),
    ("summary",        re.compile(r"^(summary|profile|objective|professional\s+summary|career\s+objective|about)\s*[:\-]?", re.I)),
    ("experience",     re.compile(r"^(experience|work\s+experience|employment|professional\s+experience|work\s+history|internship|internships)\s*[:\-]?", re.I)),
    ("education",      re.compile(r"^(education|academic\s+background|academics|schooling|qualifications)\s*[:\-]?", re.I)),
    ("skills",         re.compile(r"^(skills|technical\s+skills|core\s+competencies|competencies|technologies|tech\s+stack|languages|tools)\s*[:\-]?", re.I)),
    ("projects",       re.compile(r"^(projects?|personal\s+projects?|side\s+projects?|portfolio|open[- ]source)\s*[:\-]?", re.I)),
    ("certifications", re.compile(r"^(certifications?|certificates?|licenses?|credentials?|accreditations?)\s*[:\-]?", re.I)),
    ("awards",         re.compile(r"^(awards?|honors?|achievements?|accomplishments?|recognition)\s*[:\-]?", re.I)),
    ("publications",   re.compile(r"^(publications?|papers?|research|conferences?)\s*[:\-]?", re.I)),
    ("volunteer",      re.compile(r"^(volunteer(ing)?|community\s+service|extracurricular|activities|leadership)\s*[:\-]?", re.I)),
    ("languages",      re.compile(r"^(languages?|spoken\s+languages?|language\s+proficiency)\s*[:\-]?", re.I)),
    ("interests",      re.compile(r"^(interests?|hobbies|passions?)\s*[:\-]?", re.I)),
    ("references",     re.compile(r"^(references?|referees?)\s*[:\-]?", re.I)),
]

# A line is a "candidate header" if it:
#   - is short (≤ 60 chars)
#   - matches one of the section patterns
#   - does NOT end with punctuation like . , ; (those are body sentences)
HEADER_MAX_LEN = 60


def _classify_header(line: str) -> Optional[str]:
    """Return section type if line looks like a section header, else None."""
    stripped = line.strip()
    if not stripped or len(stripped) > HEADER_MAX_LEN:
        return None
    if stripped.endswith((".", ",", ";")):
        return None
    for section_type, pattern in SECTION_PATTERNS:
        if pattern.match(stripped):
            return section_type
    return None


def split_into_sections(text: str) -> Dict[str, str]:
    """
    Split resume text into a dict of {section_type: text}.
    Lines before the first recognized header are classified as 'contact'.
    Multiple occurrences of the same section type are concatenated.
    """
    lines = text.split("\n")
    sections: Dict[str, List[str]] = {}
    current_section = "contact"  # assume top of resume is contact block
    buffer: List[str] = []

    for line in lines:
        section_type = _classify_header(line)
        if section_type is not None:
            # Save the current buffer
            if buffer:
                sections.setdefault(current_section, []).extend(buffer)
                buffer = []
            current_section = section_type
        else:
            buffer.append(line)

    if buffer:
        sections.setdefault(current_section, []).extend(buffer)

    # Join each section's lines back to text and strip
    return {k: "\n".join(v).strip() for k, v in sections.items() if "\n".join(v).strip()}


print("Section detection helpers loaded.")

## 2 — Contact Information Extraction

In [ ]:
EMAIL_RE    = re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+")
PHONE_RE    = re.compile(r"(?:\+?1[\s.-]?)?\(?\d{3}\)?[\s.-]?\d{3}[\s.-]?\d{4}")
LINKEDIN_RE = re.compile(r"linkedin\.com/in/[\w-]+", re.I)
GITHUB_RE   = re.compile(r"github\.com/[\w-]+", re.I)


def extract_contact(contact_block: str, filename: str = "") -> Dict:
    """Pull structured contact info from the contact block."""
    emails    = EMAIL_RE.findall(contact_block)
    phones    = PHONE_RE.findall(contact_block)
    linkedins = LINKEDIN_RE.findall(contact_block)
    githubs   = GITHUB_RE.findall(contact_block)

    # Heuristic: the name is usually the first non-empty line of the contact block
    first_line = next((l.strip() for l in contact_block.split("\n") if l.strip()), "")
    # Avoid treating an email/phone as the name
    name = first_line if first_line and not EMAIL_RE.search(first_line) and not PHONE_RE.search(first_line) else ""

    return {
        "name":     name,
        "email":    emails[0] if emails else "",
        "phone":    phones[0] if phones else "",
        "linkedin": linkedins[0] if linkedins else "",
        "github":   githubs[0] if githubs else "",
    }


print("Contact extraction loaded.")

## 3 — Per-Section Parsers

Each section type gets a dedicated parser that produces structured output.

In [ ]:
DATE_RE = re.compile(
    r"(?:"
    r"(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|Jun(?:e)?|"
    r"Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)"
    r"[\s.,]*\d{4}"
    r"|\d{4}"
    r")"
    r"(?:\s*[\u2013\u2014\-–—]\s*"
        r"(?:(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|Jun(?:e)?|"
        r"Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)"
        r"[\s.,]*\d{4}"
        r"|\d{4}"
        r"|[Pp]resent|[Cc]urrent"
    r"))?",
    re.I
)

BULLET_RE = re.compile(r"^[\u2022\u2023\u2043\u204c\u204d\-\*\+>]\s+", re.M)


def _extract_dates(text: str) -> List[str]:
    return [m.group().strip() for m in DATE_RE.finditer(text) if m.group().strip()]


def parse_experience(text: str) -> List[Dict]:
    """
    Parse experience section into a list of job entries.
    Each entry: {title, company, dates, bullets, raw_text}
    """
    # Split on blank lines — each paragraph is likely one job entry
    paragraphs = re.split(r"\n{2,}", text.strip())
    entries = []
    for para in paragraphs:
        if not para.strip():
            continue
        lines = [l.strip() for l in para.split("\n") if l.strip()]
        if not lines:
            continue
        # First line: likely title | company or company | title
        header_line = lines[0]
        # Bullets: lines starting with bullet characters
        bullet_lines = [re.sub(BULLET_RE, "", l).strip() for l in lines if BULLET_RE.match(l)]
        dates = _extract_dates(para)
        entries.append({
            "raw_header": header_line,
            "dates":      dates,
            "bullets":    bullet_lines,
            "raw_text":   para.strip(),
        })
    return entries


def parse_education(text: str) -> List[Dict]:
    """
    Parse education section into a list of degree entries.
    Each entry: {institution, degree, dates, raw_text}
    """
    paragraphs = re.split(r"\n{2,}", text.strip())
    entries = []
    for para in paragraphs:
        if not para.strip():
            continue
        lines = [l.strip() for l in para.split("\n") if l.strip()]
        dates = _extract_dates(para)
        entries.append({
            "raw_header": lines[0] if lines else "",
            "dates":      dates,
            "raw_text":   para.strip(),
        })
    return entries


def parse_skills(text: str) -> List[str]:
    """
    Parse skills section into a flat list of skill tokens.
    Handles comma/pipe/bullet separated formats.
    """
    # Remove sub-headers like "Languages:" or "Frameworks:"
    cleaned = re.sub(r"^[\w\s]+:\s*", "", text, flags=re.M)
    # Split on commas, pipes, bullets, newlines
    tokens = re.split(r"[\|,\n\u2022\u2023\u2043\u204c\u204d\-]", cleaned)
    skills = [t.strip() for t in tokens if t.strip() and len(t.strip()) > 1]
    # Filter out likely non-skills (sentences)
    skills = [s for s in skills if len(s.split()) <= 5]
    return skills


print("Section parsers loaded.")

## 4 — Full Resume Parser

Combine section detection + per-section parsing into a single `parse_resume` function.

In [ ]:
def parse_resume(record: Dict) -> Dict:
    """
    Given a raw extracted record (filename, source, text, ...),
    return a structured parsed record.
    """
    text     = record.get("text", "")
    sections = split_into_sections(text)

    contact_block = sections.get("contact", "")
    contact_info  = extract_contact(contact_block, filename=record.get("filename", ""))

    parsed = {
        "candidate_id":  record.get("filename", ""),
        "source":        record.get("source", ""),
        "file_path":     record.get("file_path", ""),
        "word_count":    record.get("word_count", 0),
        "metadata":      record.get("metadata", {}),
        "contact":       contact_info,
        "sections_raw":  sections,
        "experience":    parse_experience(sections.get("experience", "")),
        "education":     parse_education(sections.get("education", "")),
        "skills":        parse_skills(sections.get("skills", "")),
        "summary":       sections.get("summary", ""),
        "projects":      parse_experience(sections.get("projects", "")),  # same structure as experience
        "certifications": [l.strip() for l in sections.get("certifications", "").split("\n") if l.strip()],
    }
    return parsed


print("Resume parser loaded.")

## 5 — Parse All Resumes

In [ ]:
with open(INPUT_PATH, "r", encoding="utf-8") as f:
    records = json.load(f)

print(f"Loaded {len(records)} resumes from {INPUT_PATH}")

parsed_records = [parse_resume(r) for r in tqdm(records, desc="Parsing")]

with open(PARSED_PATH, "w", encoding="utf-8") as f:
    json.dump(parsed_records, f, indent=2, ensure_ascii=False)

print(f"\nSaved {len(parsed_records)} parsed resumes to {PARSED_PATH}")
print(f"File size: {PARSED_PATH.stat().st_size / 1024:.1f} KB")

## 6 — Inspect a Parsed Resume

In [ ]:
sample = parsed_records[0]
print(f"Candidate : {sample['candidate_id']}")
print(f"Source    : {sample['source']}")
print(f"Contact   : {sample['contact']}")
print(f"Sections  : {list(sample['sections_raw'].keys())}")
print(f"Skills    : {sample['skills'][:10]}")
print(f"Experience entries: {len(sample['experience'])}")
if sample['experience']:
    print(f"  First entry header: {sample['experience'][0]['raw_header']}")
    print(f"  Dates: {sample['experience'][0]['dates']}")
    print(f"  Bullets: {sample['experience'][0]['bullets'][:2]}")
print(f"Education entries : {len(sample['education'])}")
if sample['education']:
    print(f"  First entry: {sample['education'][0]['raw_header']}")

## 7 — Chunking

Split each parsed resume into smaller chunks. Each chunk:
- Maps to a specific section type
- Contains focused text (one job entry, skill group, project, etc.)
- Carries metadata for filtering (candidate ID, section, source)

### Chunk structure
```json
{
  "chunk_id":      "uuid",
  "candidate_id":  "filename.pdf",
  "source":        "ds3_members",
  "section_type":  "experience",
  "text":          "...",
  "metadata":      { ... }
}
```

In [ ]:
MIN_CHUNK_CHARS = 30   # discard trivially short chunks


def make_chunk(candidate_id: str, source: str, section_type: str,
               text: str, extra_meta: Optional[Dict] = None) -> Dict:
    """Build a single chunk dict."""
    meta = {"candidate_id": candidate_id, "source": source}
    if extra_meta:
        meta.update(extra_meta)
    return {
        "chunk_id":     str(uuid.uuid4()),
        "candidate_id": candidate_id,
        "source":       source,
        "section_type": section_type,
        "text":         text.strip(),
        "metadata":     meta,
    }


def chunk_resume(parsed: Dict) -> List[Dict]:
    """
    Convert a parsed resume into a list of chunks.
    Strategy:
    - experience / projects: one chunk per entry (title + bullets concatenated)
    - education:             one chunk per entry
    - skills:                one single chunk with all skills joined
    - summary:               one chunk
    - certifications:        one chunk
    - contact:               one chunk (for name/email lookup)
    - other sections:        one chunk each (capped at 512 words)
    """
    cid    = parsed["candidate_id"]
    source = parsed["source"]
    chunks: List[Dict] = []

    # --- Contact ---
    contact = parsed.get("contact", {})
    contact_text = " | ".join(v for v in contact.values() if v)
    if len(contact_text) >= MIN_CHUNK_CHARS:
        chunks.append(make_chunk(cid, source, "contact", contact_text))

    # --- Summary ---
    summary = parsed.get("summary", "")
    if len(summary) >= MIN_CHUNK_CHARS:
        chunks.append(make_chunk(cid, source, "summary", summary))

    # --- Experience: one chunk per entry ---
    for entry in parsed.get("experience", []):
        parts = [entry["raw_header"]]
        if entry["dates"]:
            parts.append(" | ".join(entry["dates"]))
        parts.extend(entry["bullets"])
        text = "\n".join(p for p in parts if p)
        if len(text) >= MIN_CHUNK_CHARS:
            chunks.append(make_chunk(cid, source, "experience", text, {
                "dates": entry["dates"],
            }))

    # --- Education: one chunk per entry ---
    for entry in parsed.get("education", []):
        text = entry["raw_text"]
        if len(text) >= MIN_CHUNK_CHARS:
            chunks.append(make_chunk(cid, source, "education", text, {
                "dates": entry["dates"],
            }))

    # --- Skills: single chunk ---
    skills = parsed.get("skills", [])
    if skills:
        skills_text = ", ".join(skills)
        if len(skills_text) >= MIN_CHUNK_CHARS:
            chunks.append(make_chunk(cid, source, "skills", skills_text, {
                "skills_list": skills,
            }))

    # --- Projects: one chunk per entry (same shape as experience) ---
    for entry in parsed.get("projects", []):
        parts = [entry["raw_header"]]
        parts.extend(entry["bullets"])
        text = "\n".join(p for p in parts if p)
        if len(text) >= MIN_CHUNK_CHARS:
            chunks.append(make_chunk(cid, source, "projects", text))

    # --- Certifications: single chunk ---
    certs = parsed.get("certifications", [])
    if certs:
        cert_text = "\n".join(certs)
        if len(cert_text) >= MIN_CHUNK_CHARS:
            chunks.append(make_chunk(cid, source, "certifications", cert_text))

    # --- Any other raw sections not handled above ---
    handled = {"contact", "summary", "experience", "education", "skills", "projects", "certifications"}
    for section_type, section_text in parsed.get("sections_raw", {}).items():
        if section_type in handled:
            continue
        if len(section_text) >= MIN_CHUNK_CHARS:
            # Cap at 512 words to avoid monster chunks
            words = section_text.split()
            capped = " ".join(words[:512])
            chunks.append(make_chunk(cid, source, section_type, capped))

    return chunks


print("Chunking function loaded.")

## 8 — Chunk All Resumes & Save

In [ ]:
all_chunks: List[Dict] = []
for parsed in tqdm(parsed_records, desc="Chunking"):
    all_chunks.extend(chunk_resume(parsed))

with open(CHUNKS_PATH, "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, indent=2, ensure_ascii=False)

print(f"\nTotal chunks: {len(all_chunks)}")
print(f"Avg chunks per resume: {len(all_chunks) / len(parsed_records):.1f}")
print(f"Saved to: {CHUNKS_PATH}  ({CHUNKS_PATH.stat().st_size / 1024:.1f} KB)")

## 9 — Section Distribution

In [ ]:
from collections import Counter

section_counts = Counter(c["section_type"] for c in all_chunks)
print(f"{'Section':<20} {'Chunks':>8}")
print("-" * 30)
for section, count in sorted(section_counts.items(), key=lambda x: -x[1]):
    print(f"{section:<20} {count:>8}")

## 10 — Sample Chunks

In [ ]:
# Print one example chunk from each section type
seen_sections = set()
for chunk in all_chunks:
    st = chunk["section_type"]
    if st not in seen_sections:
        seen_sections.add(st)
        print(f"\n{'=' * 60}")
        print(f"SECTION : {st}")
        print(f"CANDIDATE: {chunk['candidate_id']}")
        print(f"TEXT:\n{chunk['text'][:400]}")
        print(f"METADATA: {chunk['metadata']}")
    if len(seen_sections) >= 8:
        break

## Summary

| Output | Description |
|--------|-------------|
| `resumes_parsed.json` | Per-resume structured data: contact info, experience list, education list, skills list, projects, certs |
| `resume_chunks.json` | Flat list of chunks — each with `chunk_id`, `candidate_id`, `section_type`, `text`, `metadata` |

**Next step → `03_embeddings.ipynb`:** embed each chunk's `text` field with `all-MiniLM-L6-v2` to produce chunk-level vectors for retrieval.